### Dependencies

In [2]:
%pip install stable_baselines3 crafter shimmy gymnasium imageio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.6/107.6 kB 9.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.2/187.2 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.0/268.0 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 753.1/753.1 kB 64.3 MB/s eta 0:00:00
  Created wheel for crafter: filename=crafter-1.8.3-py3-none-any.whl size=143999 sha256=e5a23da811194e12bc499dd7e1cedceb664147b8436c5dbe9d83632c0192bb05
  Stored in directory: /root/.cache/pip/wheels/2a/03/67/55cb8a55d98466fa8d880cfa81197ba78ed5a4acc0cf1f6543
Successfully built crafter


In [3]:
import gymnasium as gym
import stable_baselines3
import argparse
import crafter
from shimmy import GymV21CompatibilityV0
from stable_baselines3 import DQN
from stable_baselines3.common.vec_env import DummyVecEnv, VecMonitor
import os
import numpy as np

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


### Environment Setup

In [4]:
parser = argparse.ArgumentParser()
parser.add_argument('--steps', type=float, default=5e5)
args, _ = parser.parse_known_args()

log_path = os.path.join('Training', 'Logs')

DQN_Path = os.path.join('Training', 'Saved Models', 'DQN_Baseline_Model.zip')


### Hierarchical Wrapper (Reward shaping Improvement 1)

In [5]:
class CrafterGymnasiumWrapper(gym.Env):
    """
    Wraps crafter.Env to be compatible with Gymnasium API.
    This is the base wrapper that converts Crafter to Gymnasium.
    """

    def __init__(self):
        super().__init__()
        self.env = crafter.Env()

        # Define action and observation spaces using Gymnasium
        self.action_space = gym.spaces.Discrete(17)
        self.observation_space = gym.spaces.Box(
            low=0,
            high=255,
            shape=(64, 64, 3),
            dtype=np.uint8
        )

    def reset(self, seed=None, options=None):
        if seed is not None:
            np.random.seed(seed)
        obs = self.env.reset()
        return obs, {}

    def step(self, action):
        obs, reward, done, info = self.env.step(action)
        truncated = False
        return obs, reward, done, truncated, info

    def render(self):
        return self.env.render()

    def close(self):
        self.env.close()


In [6]:

class HierarchicalRewardWrapper(gym.Wrapper):
    """
    Improved reward shaping with tiered bonuses and no penalties.
    Only rewards NEW achievements with bonuses scaled by difficulty.
    """
    def __init__(self, env):
        super().__init__(env)
        self.previous_achievements = set()

        # Achievement tiers based on difficulty/importance
        self.achievement_tiers = {
            # Tier 1: Basic survival (small bonus)
            "collect_wood": 1.0,
            "collect_drink": 1.0,
            "collect_sapling": 1.0,
            "wake_up": 1.0,

            # Tier 2: Simple crafting/combat (medium bonus)
            "place_plant": 2.0,
            "eat_plant": 2.0,
            "place_table": 2.0,
            "defeat_zombie": 2.0,
            "eat_cow": 2.0,

            # Tier 3: Advanced crafting (larger bonus)
            "make_wood_pickaxe": 3.0,
            "make_wood_sword": 3.0,
            "collect_stone": 3.0,
            "place_stone": 3.0,
            "place_furnace": 3.0,
            "defeat_skeleton": 3.0,

            # Tier 4: Expert level (big bonus)
            "make_stone_pickaxe": 5.0,
            "make_stone_sword": 5.0,
            "collect_coal": 5.0,
            "collect_iron": 5.0,

            # Tier 5: Endgame (huge bonus)
            "make_iron_pickaxe": 10.0,
            "make_iron_sword": 10.0,
            "collect_diamond": 15.0,
        }

    def step(self, action):
        obs, reward, done, truncated, info = self.env.step(action)

        achievements = info.get("achievements", {})
        unlocked = {k for k, v in achievements.items() if v}

        # Find newly unlocked achievements
        new_achievements = unlocked - self.previous_achievements

        # Add tiered bonuses for new achievements only
        shaped_reward = reward
        for ach in new_achievements:
            bonus = self.achievement_tiers.get(ach, 1.0)
            shaped_reward += bonus

        # Update tracker
        self.previous_achievements = unlocked

        return obs, shaped_reward, done, truncated, info

    def reset(self, **kwargs):
        self.previous_achievements.clear()
        return self.env.reset(**kwargs)

### Model Building

In [10]:
class Models():
    def __init__(self):

        pass

    def baseline_dqn(self):
        """Function to create the baseline model for DQN"""
        env = DummyVecEnv([make_env])
        env = VecMonitor(env, "Training/Logs/monitor/crafter_dqn")

        # Create model and train model
        model = DQN(
            "CnnPolicy",
            env,
            learning_rate=1e-4,
            gamma=0.99,
            train_freq=4,
            buffer_size=10000,
            batch_size=32,
            target_update_interval=10000,
            exploration_fraction=0.1,
            exploration_final_eps=0.05,
            tensorboard_log=log_path,
            verbose=1
        )

        model.learn(total_timesteps=1000000)
        model.save(DQN_Path)
        print("Training complete! Model saved")

        return model, env

    def improved1_dqn():
        """Improved DQN model with hierarchical reward shaping"""
        env = DummyVecEnv([make_shaped_env])
        env = VecMonitor(env, "Training/Logs/monitor/crafter_dqn_shaped")

        model = DQN(
            "CnnPolicy",
            env,
            learning_rate=1e-3,
            gamma=0.99,
            train_freq=4,
            buffer_size=10000,
            batch_size=32,
            target_update_interval=10000,
            exploration_fraction=0.1,
            exploration_final_eps=0.05,
            tensorboard_log=log_path,
            verbose=1
        )

        model.learn(total_timesteps=1000000)
        model.save(os.path.join('Training', 'Saved Models', 'DQN_Hierarchical_Model.zip'))
        print("Training complete! Hierarchical model saved")

        return model, env

    def improved2_dqn(self):
        pass



In [8]:
def make_env():
    """Function to create a single environment instance"""
    env = crafter.Env()
    env = crafter.Recorder(
        env, './Training/Logs/jsons/',
        save_stats=True,
        save_video=False,
        save_episode=False,
    )
    env = GymV21CompatibilityV0(env=env)
    return env

def make_shaped_env():
    env = crafter.Env()
    env = crafter.Recorder(
        env, './Training/Logs/jsons_shaped/',
        save_stats=True,
        save_video=False,
        save_episode=False,
    )
    env = GymV21CompatibilityV0(env=env)
    env = HierarchicalRewardWrapper(env)
    return env


### Model Training

In [ ]:
model, env = Models.improved1_dqn()
env.close()

Using cuda device
Wrapping the env in a VecTransposeImage.
Logging to Training/Logs/DQN_1


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


----------------------------------
| rollout/            |          |
|    ep_len_mean      | 163      |
|    ep_rew_mean      | 4.1      |
|    exploration_rate | 0.994    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 95       |
|    time_elapsed     | 6        |
|    total_timesteps  | 652      |
| train/              |          |
|    learning_rate    | 0.001    |
|    loss             | 0.000381 |
|    n_updates        | 137      |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 171      |
|    ep_rew_mean      | 4.98     |
|    exploration_rate | 0.987    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 144      |
|    time_elapsed     | 9        |
|    total_timesteps  | 1369     |
| train/              |          |
|    learning_rate    | 0.001    |
|    loss             | 0.177    |
|    n_updates      

### Test Baseline DQN

In [ ]:
import imageio
import numpy as np
import json
from collections import defaultdict
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

In [ ]:
MODEL_PATH = "Training/Saved Models/DQN_Baseline_Model"  # Don't include .zip extension
NUM_EPISODES = 500
VIDEO_DIR = "./crafter_videos_baseline/"
RESULTS_DIR = "./results/"
PLOTS_DIR = "./plots_baseline/"

os.makedirs(VIDEO_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

# Define all possible achievements in Crafter for complete reporting
ALL_ACHIEVEMENTS = [
    "collect_wood", "collect_stone", "collect_coal", "collect_iron",
    "collect_diamond", "collect_sapling", "collect_drink", "place_table",
    "place_plant", "place_stone", "place_furnace", "make_wood_pickaxe",
    "make_stone_pickaxe", "make_iron_pickaxe", "make_wood_sword",
    "make_stone_sword", "make_iron_sword", "defeat_zombie", "defeat_skeleton",
     "eat_cow", "eat_plant", "wake_up"
]


def test_dqn_baseline():
    """
    Comprehensive evaluation of DQN baseline model with Windows-compatible videos.
    """
    print(f"Loading model from: {MODEL_PATH}")
    model = DQN.load(MODEL_PATH)

    # Metrics storage
    episode_rewards = []
    episode_lengths = []
    achievement_unlocks = defaultdict(int)
    achievement_per_episode = []
    action_counts = defaultdict(int)

    print(f"\nTesting DQN Baseline Model over {NUM_EPISODES} episodes...")
    print("="*70)

    for episode in range(NUM_EPISODES):
        # Use the same wrapper that was used during training for consistency
        env = CrafterGymnasiumWrapper()

        obs, info = env.reset()  # Gymnasium returns (obs, info)
        done = False
        truncated = False
        episode_reward = 0
        step = 0
        episode_achievements = set()

        # Record video for first, middle, and last episodes
        record_video = (episode == 0 or episode == NUM_EPISODES // 2 or episode == NUM_EPISODES - 1)
        episode_frames = []

        while not (done or truncated):
            # Render frame for video
            if record_video:
                frame = env.render()
                episode_frames.append(frame)

            # Predict action
            action, _ = model.predict(obs, deterministic=True)
            action_counts[int(action)] += 1

            # Gymnasium returns: obs, reward, done, truncated, info
            obs, reward, done, truncated, info = env.step(action)
            episode_reward += reward
            step += 1

            # Track achievement unlocks
            if 'achievements' in info:
                for achievement, unlocked in info['achievements'].items():
                    if unlocked and achievement not in episode_achievements:
                        achievement_unlocks[achievement] += 1
                        episode_achievements.add(achievement)

        episode_rewards.append(episode_reward)
        episode_lengths.append(step)
        achievement_per_episode.append(len(episode_achievements))

        # Save video with Windows-compatible settings
        if record_video and episode_frames:
            video_name = f"dqn_baseline_episode_{episode+1}.mp4"
            video_path = os.path.join(VIDEO_DIR, video_name)

            try:
                # Windows Media Player compatible settings
                imageio.mimsave(
                    video_path,
                    episode_frames,
                    fps=30,
                    codec='libx264',  # Use libx264 instead of h264 for better compatibility
                    quality=8,
                    pixelformat='yuv420p',  # Standard format for Windows Media Player
                )
                print(f"✓ Video saved: {video_name}")
            except Exception as e:
                print(f"✗ Error saving MP4: {e}")
                # Fallback to GIF if MP4 fails
                try:
                    gif_path = os.path.join(VIDEO_DIR, f"dqn_baseline_episode_{episode+1}.gif")
                    imageio.mimsave(gif_path, episode_frames, fps=30)
                    print(f"✓ Video saved as GIF: dqn_baseline_episode_{episode+1}.gif")
                except Exception as e2:
                    print(f"✗ Error saving GIF: {e2}")

        print(f"Episode {episode+1:3d}/{NUM_EPISODES}: "
              f"Reward={episode_reward:6.2f}, "
              f"Steps={step:4d}, "
              f"Achievements={len(episode_achievements):2d}")

        env.close()

    # Calculate metrics
    avg_reward = np.mean(episode_rewards)
    std_reward = np.std(episode_rewards)
    avg_survival_time = np.mean(episode_lengths)
    std_survival_time = np.std(episode_lengths)
    avg_achievements = np.mean(achievement_per_episode)

    # Calculate achievement unlock rates - include ALL achievements, even those with 0 unlocks
    achievement_rates = {}
    for achievement in ALL_ACHIEVEMENTS:
        count = achievement_unlocks.get(achievement, 0)
        achievement_rates[achievement] = count / NUM_EPISODES

    # Calculate geometric mean of achievement unlock rates (only for non-zero rates)
    epsilon = 1e-10
    non_zero_rates = [rate + epsilon for rate in achievement_rates.values() if rate > 0]
    if non_zero_rates:
        geometric_mean = np.exp(np.mean(np.log(non_zero_rates))) - epsilon
    else:
        geometric_mean = 0.0

    # Print summary
    print("\n" + "="*70)
    print("EVALUATION SUMMARY - DQN BASELINE")
    print("="*70)
    print(f"\nPerformance Metrics:")
    print(f"  Average Cumulative Reward:     {avg_reward:8.2f} ± {std_reward:.2f}")
    print(f"  Average Survival Time:         {avg_survival_time:8.2f} ± {std_survival_time:.2f} steps")
    print(f"  Average Achievements/Episode:  {avg_achievements:8.2f}")
    print(f"  Geometric Mean of Achievements: {geometric_mean:7.4f}")

    print(f"\nAchievement Unlock Rates (All {len(ALL_ACHIEVEMENTS)} Achievements):")
    sorted_achievements = sorted(achievement_rates.items(), key=lambda x: x[1], reverse=True)
    for achievement, rate in sorted_achievements:
        count = achievement_unlocks.get(achievement, 0)
        bar = "█" * int(rate * 50)
        bar = bar.ljust(50)  # Ensure bar is always 50 characters wide
        print(f"  {achievement:30s}: {rate*100:5.1f}% [{bar}] ({count:3d}/{NUM_EPISODES})")

    print(f"\nAction Distribution (Total actions: {sum(action_counts.values())}):")
    total_actions = sum(action_counts.values())
    for action_id in sorted(action_counts.keys()):
        count = action_counts[action_id]
        percentage = (count / total_actions) * 100
        print(f"  Action {action_id:2d}: {count:7d} ({percentage:5.2f}%)")

    # Create visualizations
    print("\nGenerating visualizations...")
    create_visualizations(
        episode_rewards,
        episode_lengths,
        achievement_per_episode,
        achievement_rates,
        action_counts
    )

    # Save results to JSON
    results = {
        "model": "DQN_Baseline",
        "num_episodes": NUM_EPISODES,
        "timestamp": datetime.now().isoformat(),
        "metrics": {
            "average_reward": float(avg_reward),
            "std_reward": float(std_reward),
            "average_survival_time": float(avg_survival_time),
            "std_survival_time": float(std_survival_time),
            "average_achievements_per_episode": float(avg_achievements),
            "geometric_mean_achievements": float(geometric_mean),
            "total_unique_achievements": len([a for a in achievement_rates.values() if a > 0]),
            "total_possible_achievements": len(ALL_ACHIEVEMENTS)
        },
        "achievement_unlock_rates": achievement_rates,
        "action_distribution": {k: int(v) for k, v in action_counts.items()},
        "episode_rewards": episode_rewards,
        "episode_lengths": episode_lengths,
        "achievements_per_episode": achievement_per_episode
    }

    results_path = os.path.join(RESULTS_DIR, "dqn_baseline_results.json")
    with open(results_path, 'w') as f:
        json.dump(results, f, indent=2)

    print(f"✓ Results saved to: {results_path}")
    print("="*70)

    return results


def create_visualizations(episode_rewards, episode_lengths, achievements_per_episode,
                         achievement_rates, action_counts):
    """Create comprehensive visualization plots"""

    sns.set_style("whitegrid")

    # 1. Rewards and Survival Time over Episodes
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))

    # Rewards over episodes
    axes[0, 0].plot(episode_rewards, linewidth=1, alpha=0.7, color='blue')
    axes[0, 0].axhline(np.mean(episode_rewards), color='red', linestyle='--',
                       label=f'Mean: {np.mean(episode_rewards):.2f}')
    axes[0, 0].set_xlabel('Episode')
    axes[0, 0].set_ylabel('Cumulative Reward')
    axes[0, 0].set_title('Episode Rewards - Baseline Model')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    # Survival time over episodes
    axes[0, 1].plot(episode_lengths, linewidth=1, alpha=0.7, color='green')
    axes[0, 1].axhline(np.mean(episode_lengths), color='red', linestyle='--',
                       label=f'Mean: {np.mean(episode_lengths):.2f}')
    axes[0, 1].set_xlabel('Episode')
    axes[0, 1].set_ylabel('Steps')
    axes[0, 1].set_title('Survival Time per Episode - Baseline Model')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    # Achievements per episode
    axes[1, 0].plot(achievements_per_episode, linewidth=1, alpha=0.7, color='purple')
    axes[1, 0].axhline(np.mean(achievements_per_episode), color='red', linestyle='--',
                       label=f'Mean: {np.mean(achievements_per_episode):.2f}')
    axes[1, 0].set_xlabel('Episode')
    axes[1, 0].set_ylabel('Number of Achievements')
    axes[1, 0].set_title('Achievements Unlocked per Episode - Baseline Model')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    # Distribution of rewards
    axes[1, 1].hist(episode_rewards, bins=20, color='blue', alpha=0.7, edgecolor='black')
    axes[1, 1].axvline(np.mean(episode_rewards), color='red', linestyle='--',
                       label=f'Mean: {np.mean(episode_rewards):.2f}')
    axes[1, 1].set_xlabel('Cumulative Reward')
    axes[1, 1].set_ylabel('Frequency')
    axes[1, 1].set_title('Distribution of Episode Rewards - Baseline Model')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, 'episode_metrics_baseline.png'), dpi=300, bbox_inches='tight')
    print(f"✓ Saved: episode_metrics_baseline.png")
    plt.close()

    # 2. Achievement unlock rates - show ALL achievements
    fig, ax = plt.subplots(figsize=(12, max(6, len(achievement_rates) * 0.3)))
    sorted_achievements = sorted(achievement_rates.items(), key=lambda x: x[1], reverse=True)
    achievements, rates = zip(*sorted_achievements)

    # Create color array: blue for unlocked, light gray for 0%
    colors = ['skyblue' if rate > 0 else 'lightgray' for rate in rates]

    bars = ax.barh(range(len(achievements)), [r * 100 for r in rates], color=colors, edgecolor='black')
    ax.set_yticks(range(len(achievements)))
    ax.set_yticklabels(achievements)
    ax.set_xlabel('Unlock Rate (%)')
    ax.set_title('Achievement Unlock Rates - Baseline Model (All Achievements)')
    ax.grid(axis='x', alpha=0.3)

    # Add percentage labels
    for i, (bar, rate) in enumerate(zip(bars, rates)):
        if rate > 0:
            ax.text(rate * 100 + 1, i, f'{rate*100:.1f}%', va='center', fontweight='bold')
        else:
            ax.text(1, i, '0.0%', va='center', color='gray', fontstyle='italic')

    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, 'achievement_rates_baseline.png'), dpi=300, bbox_inches='tight')
    print(f"✓ Saved: achievement_rates_baseline.png")
    plt.close()

    # 3. Action distribution
    if action_counts:
        fig, ax = plt.subplots(figsize=(10, 6))
        actions = sorted(action_counts.keys())
        counts = [action_counts[a] for a in actions]
        total = sum(counts)
        percentages = [c / total * 100 for c in counts]

        bars = ax.bar(actions, percentages, color='coral', edgecolor='black')
        ax.set_xlabel('Action ID')
        ax.set_ylabel('Percentage (%)')
        ax.set_title('Action Distribution - Baseline Model')
        ax.grid(axis='y', alpha=0.3)

        # Add percentage labels on bars
        for bar, pct in zip(bars, percentages):
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{pct:.1f}%', ha='center', va='bottom', fontsize=8)

        plt.tight_layout()
        plt.savefig(os.path.join(PLOTS_DIR, 'action_distribution_baseline.png'), dpi=300, bbox_inches='tight')
        print(f"✓ Saved: action_distribution_baseline.png")
        plt.close()


In [ ]:
if __name__ == "__main__":
    try:
        results = test_dqn_baseline()
        print("\n✓ Testing completed successfully!")
    except FileNotFoundError:
        print(f"\n✗ Error: Model file not found at {MODEL_PATH}")
        print("Please ensure the model has been trained and saved to the correct location.")
    except Exception as e:
        print(f"\n✗ Error during testing: {str(e)}")
        import traceback
        traceback.print_exc()

### Test Improvement 1 DQN

In [ ]:
MODEL_PATH = "Training/Saved Models/DQN_Hierarchical_Model"
NUM_EPISODES = 500
VIDEO_DIR = "./crafter_videos_improved1/"
RESULTS_DIR = "./results/"
PLOTS_DIR = "./plots_improved1/"

os.makedirs(VIDEO_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

# Define all possible achievements in Crafter for complete reporting
ALL_ACHIEVEMENTS = [
    "collect_wood", "collect_stone", "collect_coal", "collect_iron",
    "collect_diamond", "collect_sapling", "collect_drink", "place_table",
    "place_plant", "place_stone", "place_furnace", "make_wood_pickaxe",
    "make_stone_pickaxe", "make_iron_pickaxe", "make_wood_sword",
    "make_stone_sword", "make_iron_sword", "defeat_zombie", "defeat_skeleton",
    "eat_cow", "eat_plant", "wake_up"
]


def test_dqn_improved1():
    """
    Comprehensive evaluation of DQN improved model with hierarchical reward shaping:
    - Achievement unlock rates and progression
    - Geometric mean of achievement unlock rates
    - Survival time (timesteps per episode)
    - Cumulative reward per episode
    - Action distribution analysis
    - Episode-by-episode progression
    """
    print(f"Loading model from: {MODEL_PATH}")
    model = DQN.load(MODEL_PATH)

    # Metrics storage
    episode_rewards = []
    episode_lengths = []
    achievement_unlocks = defaultdict(int)
    achievement_per_episode = []
    action_counts = defaultdict(int)

    print(f"\nTesting DQN Improved Model (Hierarchical Reward Shaping) over {NUM_EPISODES} episodes...")
    print("="*70)

    for episode in range(NUM_EPISODES):
        # Create environment using your Gymnasium wrapper
        env = CrafterGymnasiumWrapper()
        env = HierarchicalRewardWrapper(env)

        obs, info = env.reset()
        done = False
        truncated = False
        episode_reward = 0
        step = 0
        episode_achievements = set()

        # Record video for first, middle, and last episodes
        record_video = (episode == 0 or episode == NUM_EPISODES // 2 or episode == NUM_EPISODES - 1)
        episode_frames = []

        while not (done or truncated):
            # Render frame for video
            if record_video:
                frame = env.render()
                episode_frames.append(frame)

            # Predict action
            action, _ = model.predict(obs, deterministic=True)
            action_counts[int(action)] += 1

            obs, reward, done, truncated, info = env.step(action)

            episode_reward += reward
            step += 1

            # Track achievement unlocks
            if 'achievements' in info:
                for achievement, unlocked in info['achievements'].items():
                    if unlocked and achievement not in episode_achievements:
                        achievement_unlocks[achievement] += 1
                        episode_achievements.add(achievement)

        episode_rewards.append(episode_reward)
        episode_lengths.append(step)
        achievement_per_episode.append(len(episode_achievements))

        # Save video for recorded episodes with Windows-compatible settings
        if record_video and episode_frames:
            video_name = f"dqn_improved1_episode_{episode+1}.mp4"
            video_path = os.path.join(VIDEO_DIR, video_name)

            # Windows-compatible video settings
            try:
                imageio.mimsave(
                    video_path,
                    episode_frames,
                    fps=30,
                    codec='libx264',  # More compatible codec
                    quality=8,
                    pixelformat='yuv420p',  # Standard format compatible with Windows Media Player
                )
                print(f"✓ Video saved: {video_name}")
            except Exception as e:
                print(f"✗ Error saving video: {e}")
                # Fallback to GIF if MP4 fails
                try:
                    gif_path = os.path.join(VIDEO_DIR, f"dqn_improved1_episode_{episode+1}.gif")
                    imageio.mimsave(gif_path, episode_frames, fps=30)
                    print(f"✓ Video saved as GIF: dqn_improved1_episode_{episode+1}.gif")
                except Exception as e2:
                    print(f"✗ Error saving GIF: {e2}")

        print(f"Episode {episode+1:3d}/{NUM_EPISODES}: "
              f"Reward={episode_reward:6.2f}, "
              f"Steps={step:4d}, "
              f"Achievements={len(episode_achievements):2d}")

        env.close()

    # Calculate metrics
    avg_reward = np.mean(episode_rewards)
    std_reward = np.std(episode_rewards)
    avg_survival_time = np.mean(episode_lengths)
    std_survival_time = np.std(episode_lengths)
    avg_achievements = np.mean(achievement_per_episode)

    # Calculate achievement unlock rates - include ALL achievements, even those with 0 unlocks
    achievement_rates = {}
    for achievement in ALL_ACHIEVEMENTS:
        count = achievement_unlocks.get(achievement, 0)
        achievement_rates[achievement] = count / NUM_EPISODES

    # Calculate geometric mean of achievement unlock rates (only for non-zero rates)
    epsilon = 1e-10
    non_zero_rates = [rate + epsilon for rate in achievement_rates.values() if rate > 0]
    if non_zero_rates:
        geometric_mean = np.exp(np.mean(np.log(non_zero_rates))) - epsilon
    else:
        geometric_mean = 0.0

    # Print summary
    print("\n" + "="*70)
    print("EVALUATION SUMMARY - DQN IMPROVED (HIERARCHICAL REWARD SHAPING)")
    print("="*70)
    print(f"\nPerformance Metrics:")
    print(f"  Average Cumulative Reward:     {avg_reward:8.2f} ± {std_reward:.2f}")
    print(f"  Average Survival Time:         {avg_survival_time:8.2f} ± {std_survival_time:.2f} steps")
    print(f"  Average Achievements/Episode:  {avg_achievements:8.2f}")
    print(f"  Geometric Mean of Achievements: {geometric_mean:7.4f}")

    print(f"\nAchievement Unlock Rates (All {len(ALL_ACHIEVEMENTS)} Achievements):")
    sorted_achievements = sorted(achievement_rates.items(), key=lambda x: x[1], reverse=True)
    for achievement, rate in sorted_achievements:
        count = achievement_unlocks.get(achievement, 0)
        bar = "█" * int(rate * 50)
        bar = bar.ljust(50)  # Ensure bar is always 50 characters wide
        print(f"  {achievement:30s}: {rate*100:5.1f}% [{bar}] ({count:3d}/{NUM_EPISODES})")

    print(f"\nAction Distribution (Total actions: {sum(action_counts.values())}):")
    total_actions = sum(action_counts.values())
    for action_id in sorted(action_counts.keys()):
        count = action_counts[action_id]
        percentage = (count / total_actions) * 100
        print(f"  Action {action_id:2d}: {count:7d} ({percentage:5.2f}%)")

    # Create visualizations
    print("\nGenerating visualizations...")
    create_visualizations(
        episode_rewards,
        episode_lengths,
        achievement_per_episode,
        achievement_rates,
        action_counts
    )

    # Save results to JSON
    results = {
        "model": "DQN_Improved1_Hierarchical",
        "num_episodes": NUM_EPISODES,
        "timestamp": datetime.now().isoformat(),
        "metrics": {
            "average_reward": float(avg_reward),
            "std_reward": float(std_reward),
            "average_survival_time": float(avg_survival_time),
            "std_survival_time": float(std_survival_time),
            "average_achievements_per_episode": float(avg_achievements),
            "geometric_mean_achievements": float(geometric_mean),
            "total_unique_achievements": len([a for a in achievement_rates.values() if a > 0]),
            "total_possible_achievements": len(ALL_ACHIEVEMENTS)
        },
        "achievement_unlock_rates": achievement_rates,
        "action_distribution": {k: int(v) for k, v in action_counts.items()},
        "episode_rewards": episode_rewards,
        "episode_lengths": episode_lengths,
        "achievements_per_episode": achievement_per_episode
    }

    results_path = os.path.join(RESULTS_DIR, "dqn_improved1_results.json")
    with open(results_path, 'w') as f:
        json.dump(results, f, indent=2)

    print(f"✓ Results saved to: {results_path}")
    print("="*70)

    return results


def create_visualizations(episode_rewards, episode_lengths, achievements_per_episode,
                         achievement_rates, action_counts):
    """Create comprehensive visualization plots"""

    sns.set_style("whitegrid")

    # 1. Rewards and Survival Time over Episodes
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))

    # Rewards over episodes
    axes[0, 0].plot(episode_rewards, linewidth=1, alpha=0.7, color='blue')
    axes[0, 0].axhline(np.mean(episode_rewards), color='red', linestyle='--',
                       label=f'Mean: {np.mean(episode_rewards):.2f}')
    axes[0, 0].set_xlabel('Episode')
    axes[0, 0].set_ylabel('Cumulative Reward')
    axes[0, 0].set_title('Episode Rewards - Improved Model')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    # Survival time over episodes
    axes[0, 1].plot(episode_lengths, linewidth=1, alpha=0.7, color='green')
    axes[0, 1].axhline(np.mean(episode_lengths), color='red', linestyle='--',
                       label=f'Mean: {np.mean(episode_lengths):.2f}')
    axes[0, 1].set_xlabel('Episode')
    axes[0, 1].set_ylabel('Steps')
    axes[0, 1].set_title('Survival Time per Episode - Improved Model')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    # Achievements per episode
    axes[1, 0].plot(achievements_per_episode, linewidth=1, alpha=0.7, color='purple')
    axes[1, 0].axhline(np.mean(achievements_per_episode), color='red', linestyle='--',
                       label=f'Mean: {np.mean(achievements_per_episode):.2f}')
    axes[1, 0].set_xlabel('Episode')
    axes[1, 0].set_ylabel('Number of Achievements')
    axes[1, 0].set_title('Achievements Unlocked per Episode - Improved Model')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    # Distribution of rewards
    axes[1, 1].hist(episode_rewards, bins=20, color='blue', alpha=0.7, edgecolor='black')
    axes[1, 1].axvline(np.mean(episode_rewards), color='red', linestyle='--',
                       label=f'Mean: {np.mean(episode_rewards):.2f}')
    axes[1, 1].set_xlabel('Cumulative Reward')
    axes[1, 1].set_ylabel('Frequency')
    axes[1, 1].set_title('Distribution of Episode Rewards - Improved Model')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, 'episode_metrics_improved1.png'), dpi=300, bbox_inches='tight')
    print(f"✓ Saved: episode_metrics_improved1.png")
    plt.close()

    # 2. Achievement unlock rates - show ALL achievements
    fig, ax = plt.subplots(figsize=(12, max(6, len(achievement_rates) * 0.3)))
    sorted_achievements = sorted(achievement_rates.items(), key=lambda x: x[1], reverse=True)
    achievements, rates = zip(*sorted_achievements)

    # Create color array: blue for unlocked, light gray for 0%
    colors = ['skyblue' if rate > 0 else 'lightgray' for rate in rates]

    bars = ax.barh(range(len(achievements)), [r * 100 for r in rates], color=colors, edgecolor='black')
    ax.set_yticks(range(len(achievements)))
    ax.set_yticklabels(achievements)
    ax.set_xlabel('Unlock Rate (%)')
    ax.set_title('Achievement Unlock Rates - Improved Model (All Achievements)')
    ax.grid(axis='x', alpha=0.3)

    # Add percentage labels
    for i, (bar, rate) in enumerate(zip(bars, rates)):
        if rate > 0:
            ax.text(rate * 100 + 1, i, f'{rate*100:.1f}%', va='center', fontweight='bold')
        else:
            ax.text(1, i, '0.0%', va='center', color='gray', fontstyle='italic')

    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, 'achievement_rates_improved1.png'), dpi=300, bbox_inches='tight')
    print(f"✓ Saved: achievement_rates_improved1.png")
    plt.close()

    # 3. Action distribution
    if action_counts:
        fig, ax = plt.subplots(figsize=(10, 6))
        actions = sorted(action_counts.keys())
        counts = [action_counts[a] for a in actions]
        total = sum(counts)
        percentages = [c / total * 100 for c in counts]

        bars = ax.bar(actions, percentages, color='coral', edgecolor='black')
        ax.set_xlabel('Action ID')
        ax.set_ylabel('Percentage (%)')
        ax.set_title('Action Distribution - Improved Model')
        ax.grid(axis='y', alpha=0.3)

        # Add percentage labels on bars
        for bar, pct in zip(bars, percentages):
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{pct:.1f}%', ha='center', va='bottom', fontsize=8)

        plt.tight_layout()
        plt.savefig(os.path.join(PLOTS_DIR, 'action_distribution_improved1.png'), dpi=300, bbox_inches='tight')
        print(f"✓ Saved: action_distribution_improved1.png")
        plt.close()

In [ ]:
if __name__ == "__main__":
    try:
        results = test_dqn_improved1()
        print("\n✓ Testing completed successfully!")
    except FileNotFoundError:
        print(f"\n✗ Error: Model file not found at {MODEL_PATH}")
        print("Please ensure the model has been trained and saved to the correct location.")
    except Exception as e:
        print(f"\n✗ Error during testing: {str(e)}")
        import traceback
        traceback.print_exc()

### Model Comparison

In [ ]:
RESULTS_DIR = "./results/"
COMPARISON_DIR = "./comparison_plots/"

os.makedirs(COMPARISON_DIR, exist_ok=True)

def load_results(filename):
    """Load results from JSON file"""
    filepath = os.path.join(RESULTS_DIR, filename)
    try:
        with open(filepath, 'r') as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"Warning: {filename} not found")
        return None


def compare_models():
    """Compare baseline and improved DQN models"""

    # Load results
    baseline_results = load_results("dqn_baseline_results.json")
    improved1_results = load_results("dqn_improved1_results.json")

    if not baseline_results or not improved1_results:
        print("Error: Missing results files. Please run both test scripts first.")
        print("Expected files:")
        print(f"  - {os.path.join(RESULTS_DIR, 'dqn_baseline_results.json')}")
        print(f"  - {os.path.join(RESULTS_DIR, 'dqn_improved1_results.json')}")
        return

    print("="*80)
    print("MODEL COMPARISON: DQN BASELINE vs DQN IMPROVED (HIERARCHICAL REWARD SHAPING)")
    print("="*80)

    # Extract metrics
    baseline_metrics = baseline_results['metrics']
    improved1_metrics = improved1_results['metrics']

    # Print comparison table
    print("\n{:<40s} {:>15s} {:>15s} {:>10s}".format(
        "Metric", "Baseline", "Improved", "Change"
    ))
    print("-"*80)

    metrics_to_compare = [
        ("average_reward", "Avg Cumulative Reward", ".2f"),
        ("average_survival_time", "Avg Survival Time", ".2f"),
        ("average_achievements_per_episode", "Avg Achievements/Episode", ".2f"),
        ("geometric_mean_achievements", "Geometric Mean (Achievements)", ".4f"),
        ("total_unique_achievements", "Total Unique Achievements", "d")
    ]

    for metric_key, metric_name, fmt in metrics_to_compare:
        baseline_val = baseline_metrics[metric_key]
        improved_val = improved1_metrics[metric_key]

        if isinstance(baseline_val, (int, float)) and baseline_val != 0:
            change_pct = ((improved_val - baseline_val) / abs(baseline_val)) * 100
            change_str = f"{change_pct:+.1f}%"
        else:
            change_str = "N/A"

        print(f"{metric_name:<40s} {baseline_val:>15{fmt}} {improved_val:>15{fmt}} {change_str:>10s}")

    # Create comparison visualizations
    print("\n" + "="*80)
    print("Generating comparison plots...")

    sns.set_style("whitegrid")
    plt.rcParams['font.size'] = 10

    # 1. Side-by-side metric comparison
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))

    models = ['Baseline', 'Improved']
    colors = ['#1f77b4', '#ff7f0e']  # Better color scheme

    # Average rewards
    rewards = [baseline_metrics['average_reward'], improved1_metrics['average_reward']]
    reward_stds = [baseline_metrics['std_reward'], improved1_metrics['std_reward']]
    bars = axes[0, 0].bar(models, rewards, yerr=reward_stds, capsize=5,
                         color=colors, edgecolor='black', alpha=0.8)
    axes[0, 0].set_ylabel('Average Cumulative Reward')
    axes[0, 0].set_title('Average Cumulative Reward Comparison')
    axes[0, 0].grid(axis='y', alpha=0.3)
    for bar, val in zip(bars, rewards):
        height = bar.get_height()
        axes[0, 0].text(bar.get_x() + bar.get_width()/2., height + 0.01,
                       f'{val:.2f}', ha='center', va='bottom', fontweight='bold')

    # Average survival time
    survival = [baseline_metrics['average_survival_time'], improved1_metrics['average_survival_time']]
    survival_stds = [baseline_metrics['std_survival_time'], improved1_metrics['std_survival_time']]
    bars = axes[0, 1].bar(models, survival, yerr=survival_stds, capsize=5,
                         color=colors, edgecolor='black', alpha=0.8)
    axes[0, 1].set_ylabel('Average Survival Time (steps)')
    axes[0, 1].set_title('Average Survival Time Comparison')
    axes[0, 1].grid(axis='y', alpha=0.3)
    for bar, val in zip(bars, survival):
        height = bar.get_height()
        axes[0, 1].text(bar.get_x() + bar.get_width()/2., height + 5,
                       f'{val:.0f}', ha='center', va='bottom', fontweight='bold')

    # Average achievements per episode
    achievements = [baseline_metrics['average_achievements_per_episode'],
                   improved1_metrics['average_achievements_per_episode']]
    bars = axes[1, 0].bar(models, achievements, color=colors, edgecolor='black', alpha=0.8)
    axes[1, 0].set_ylabel('Average Achievements per Episode')
    axes[1, 0].set_title('Average Achievements Comparison')
    axes[1, 0].grid(axis='y', alpha=0.3)
    for bar, val in zip(bars, achievements):
        height = bar.get_height()
        axes[1, 0].text(bar.get_x() + bar.get_width()/2., height + 0.01,
                       f'{val:.2f}', ha='center', va='bottom', fontweight='bold')

    # Geometric mean
    geom_means = [baseline_metrics['geometric_mean_achievements'],
                  improved1_metrics['geometric_mean_achievements']]
    bars = axes[1, 1].bar(models, geom_means, color=colors, edgecolor='black', alpha=0.8)
    axes[1, 1].set_ylabel('Geometric Mean')
    axes[1, 1].set_title('Geometric Mean of Achievement Rates')
    axes[1, 1].grid(axis='y', alpha=0.3)
    for bar, val in zip(bars, geom_means):
        height = bar.get_height()
        axes[1, 1].text(bar.get_x() + bar.get_width()/2., height + 0.001,
                       f'{val:.4f}', ha='center', va='bottom', fontweight='bold')

    plt.tight_layout()
    plt.savefig(os.path.join(COMPARISON_DIR, 'metrics_comparison.png'), dpi=300, bbox_inches='tight')
    print(f"✓ Saved: metrics_comparison.png")
    plt.close()

    # 2. Episode rewards over time comparison
    fig, ax = plt.subplots(figsize=(15, 6))

    baseline_rewards = baseline_results['episode_rewards']
    improved1_rewards = improved1_results['episode_rewards']

    # Plot with moving average for clarity
    window = min(20, len(baseline_rewards) // 10)  # Adaptive window size
    if window < 2:
        window = 2

    baseline_ma = np.convolve(baseline_rewards, np.ones(window)/window, mode='valid')
    improved1_ma = np.convolve(improved1_rewards, np.ones(window)/window, mode='valid')

    # Plot raw data with low alpha
    ax.plot(baseline_rewards, alpha=0.15, color=colors[0], linewidth=0.8, label='Baseline (Raw)')
    ax.plot(improved1_rewards, alpha=0.15, color=colors[1], linewidth=0.8, label='Improved (Raw)')

    # Plot moving averages
    ax.plot(range(window-1, len(baseline_rewards)), baseline_ma,
            label=f'Baseline ({window}-ep MA)', color=colors[0], linewidth=2.5)
    ax.plot(range(window-1, len(improved1_rewards)), improved1_ma,
            label=f'Improved ({window}-ep MA)', color=colors[1], linewidth=2.5)

    # Mean lines
    ax.axhline(np.mean(baseline_rewards), color=colors[0], linestyle='--', alpha=0.7,
               label=f'Baseline Mean: {np.mean(baseline_rewards):.2f}')
    ax.axhline(np.mean(improved1_rewards), color=colors[1], linestyle='--', alpha=0.7,
               label=f'Improved Mean: {np.mean(improved1_rewards):.2f}')

    ax.set_xlabel('Episode')
    ax.set_ylabel('Cumulative Reward')
    ax.set_title('Episode Rewards Comparison (with moving average)')
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(COMPARISON_DIR, 'rewards_over_time.png'), dpi=300, bbox_inches='tight')
    print(f"✓ Saved: rewards_over_time.png")
    plt.close()

    # 3. Achievement unlock rates comparison
    baseline_achievements = baseline_results['achievement_unlock_rates']
    improved1_achievements = improved1_results['achievement_unlock_rates']

    # Get all unique achievements
    all_achievements = set(baseline_achievements.keys()) | set(improved1_achievements.keys())

    if all_achievements:
        fig, ax = plt.subplots(figsize=(14, max(8, len(all_achievements) * 0.4)))

        # Sort by improved model performance for better visualization
        achievements_sorted = sorted(all_achievements,
                                   key=lambda x: improved1_achievements.get(x, 0),
                                   reverse=True)
        y_pos = np.arange(len(achievements_sorted))

        baseline_rates = [baseline_achievements.get(ach, 0) * 100 for ach in achievements_sorted]
        improved1_rates = [improved1_achievements.get(ach, 0) * 100 for ach in achievements_sorted]

        bar_height = 0.35

        bars1 = ax.barh(y_pos - bar_height/2, baseline_rates, bar_height,
                       label='Baseline', color=colors[0], edgecolor='black', alpha=0.8)
        bars2 = ax.barh(y_pos + bar_height/2, improved1_rates, bar_height,
                       label='Improved', color=colors[1], edgecolor='black', alpha=0.8)

        ax.set_yticks(y_pos)
        ax.set_yticklabels(achievements_sorted)
        ax.set_xlabel('Unlock Rate (%)')
        ax.set_title('Achievement Unlock Rates Comparison')
        ax.legend()
        ax.grid(axis='x', alpha=0.3)

        # Add value labels
        for bars in [bars1, bars2]:
            for bar in bars:
                width = bar.get_width()
                if width > 0:
                    ax.text(width + 0.5, bar.get_y() + bar.get_height()/2.,
                           f'{width:.1f}%', ha='left', va='center', fontsize=8)

        plt.tight_layout()
        plt.savefig(os.path.join(COMPARISON_DIR, 'achievement_comparison.png'), dpi=300, bbox_inches='tight')
        print(f"✓ Saved: achievement_comparison.png")
        plt.close()

    # 4. Survival time distribution
    fig, ax = plt.subplots(figsize=(12, 6))

    baseline_lengths = baseline_results['episode_lengths']
    improved1_lengths = improved1_results['episode_lengths']

    # Use density for better comparison
    ax.hist(baseline_lengths, bins=30, alpha=0.6, label='Baseline',
            color=colors[0], edgecolor='black', density=True)
    ax.hist(improved1_lengths, bins=30, alpha=0.6, label='Improved',
            color=colors[1], edgecolor='black', density=True)

    ax.axvline(np.mean(baseline_lengths), color=colors[0], linestyle='--', linewidth=2,
               label=f'Baseline Mean: {np.mean(baseline_lengths):.0f}')
    ax.axvline(np.mean(improved1_lengths), color=colors[1], linestyle='--', linewidth=2,
               label=f'Improved Mean: {np.mean(improved1_lengths):.0f}')

    ax.set_xlabel('Episode Length (steps)')
    ax.set_ylabel('Density')
    ax.set_title('Survival Time Distribution Comparison')
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(COMPARISON_DIR, 'survival_distribution.png'), dpi=300, bbox_inches='tight')
    print(f"✓ Saved: survival_distribution.png")
    plt.close()

    # 5. Additional: Achievements progression over episodes
    fig, ax = plt.subplots(figsize=(15, 6))

    baseline_achievements_ep = baseline_results['achievements_per_episode']
    improved1_achievements_ep = improved1_results['achievements_per_episode']

    # Cumulative achievements
    baseline_cumulative = np.cumsum(baseline_achievements_ep)
    improved1_cumulative = np.cumsum(improved1_achievements_ep)

    ax.plot(baseline_cumulative, label='Baseline (Cumulative)', color=colors[0], linewidth=2)
    ax.plot(improved1_cumulative, label='Improved (Cumulative)', color=colors[1], linewidth=2)

    # Moving average of achievements per episode
    window_ach = min(10, len(baseline_achievements_ep) // 20)
    if window_ach < 2:
        window_ach = 2

    baseline_ach_ma = np.convolve(baseline_achievements_ep, np.ones(window_ach)/window_ach, mode='valid')
    improved1_ach_ma = np.convolve(improved1_achievements_ep, np.ones(window_ach)/window_ach, mode='valid')

    ax.plot(range(window_ach-1, len(baseline_achievements_ep)), baseline_ach_ma,
            label=f'Baseline ({window_ach}-ep MA)', color=colors[0], linestyle='--', alpha=0.7)
    ax.plot(range(window_ach-1, len(improved1_achievements_ep)), improved1_ach_ma,
            label=f'Improved ({window_ach}-ep MA)', color=colors[1], linestyle='--', alpha=0.7)

    ax.set_xlabel('Episode')
    ax.set_ylabel('Achievements')
    ax.set_title('Achievements Progression Over Episodes')
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(COMPARISON_DIR, 'achievements_progression.png'), dpi=300, bbox_inches='tight')
    print(f"✓ Saved: achievements_progression.png")
    plt.close()

    print("\n" + "="*80)
    print("✓ Comparison complete! All plots saved to:", COMPARISON_DIR)
    print("="*80)

    # Print summary insights
    print("\nKEY INSIGHTS:")
    print("-" * 40)

    reward_improvement = ((improved1_metrics['average_reward'] - baseline_metrics['average_reward'])
                         / abs(baseline_metrics['average_reward'])) * 100
    survival_improvement = ((improved1_metrics['average_survival_time'] - baseline_metrics['average_survival_time'])
                           / abs(baseline_metrics['average_survival_time'])) * 100
    achievements_improvement = ((improved1_metrics['average_achievements_per_episode'] - baseline_metrics['average_achievements_per_episode'])
                               / abs(baseline_metrics['average_achievements_per_episode'])) * 100

    print(f"• Reward improvement: {reward_improvement:+.1f}%")
    print(f"• Survival time improvement: {survival_improvement:+.1f}%")
    print(f"• Achievements improvement: {achievements_improvement:+.1f}%")
    print(f"• Unique achievements: {baseline_metrics['total_unique_achievements']} → {improved1_metrics['total_unique_achievements']}")


if __name__ == "__main__":
    compare_models()